In [1]:
!nvidia-smi

Fri Oct 16 03:57:06 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import userdata
from huggingface_hub import login, whoami

login(token=userdata.get("HF_TOKEN"))
print(whoami()["name"])

txchrixo


In [3]:
!pip install -q -U "transformers>=4.57" accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 140.4 MB/s eta 0:00:00


In [4]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/medgemma-1.5-4b-it"
# MODEL_ID = "Qwen/Qwen3-VL-4B-Instruct"

a = torch.randn(256, 256, device="cuda", dtype=torch.bfloat16)
print("bfloat16 ok:", torch.isfinite((a @ a).float()).all().item())

model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="auto").eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(torch.cuda.get_device_name(0), f"{torch.cuda.memory_allocated() / 1e9:.1f} GB used")

bfloat16 ok: True


config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

NVIDIA L4 8.6 GB used


In [5]:
import time

def ask(text, image=None, max_new_tokens=1500):
    content = [{"type": "text", "text": text}]
    if image is not None:
        content.append({"type": "image", "image": image})
    inputs = processor.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True,
                                           tokenize=True, return_dict=True, return_tensors="pt").to(model.device)
    if "pixel_values" in inputs:
        inputs["pixel_values"] = inputs["pixel_values"].to(torch.bfloat16)
    n = inputs["input_ids"].shape[-1]
    t = time.time()
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    print(f"({time.time() - t:.0f} s, {len(out[0]) - n} tokens)")
    return processor.decode(out[0][n:], skip_special_tokens=True).strip()

print(ask("Réponds en une phrase : quelle est la capitale du Cameroun ?"))

(2 s, 11 tokens)
La capitale du Cameroun est Yaoundé.


In [6]:
import io
from google.colab import files
from PIL import Image

uploaded = files.upload()
images = {name: Image.open(io.BytesIO(data)).convert("RGB") for name, data in uploaded.items()}
print(list(images))

Saving R001_base.png to R001_base.png
['R001_base.png']


In [7]:
QUESTION = "Voici le résultat d'analyse de mon oncle. Qu'est-ce que ça veut dire ? Est-ce que c'est grave ?"

for name, img in images.items():
    print("=" * 80)
    print(name)
    print(ask(QUESTION, img))

R001_base.png
(36 s, 558 tokens)
Bonjour,

Je ne peux pas interpréter directement les résultats d'analyse pour vous. L'interprétation doit être faite par un professionnel de santé (médecin, pharmacien, etc.) qui connaît l'historique médical complet de votre oncle, son âge, son état de santé général et d'autres facteurs.

Cependant, je peux vous donner une information générale sur ce que les résultats que vous avez présentés pourraient signifier :

*   **PSA total :** Le PSA (Antigéné Prostate Spécifique) est une protéine produite par les cellules de la prostate. Son taux sanguin peut être mesuré pour aider à détecter certains types de cancer de la prostate.
*   **Résultat : 3,80 ng/mL**
*   **Valeurs de référence : < 4,00 ng/mL**

**Ce que cela signifie généralement :**

*   Un taux de PSA total supérieur à 4,00 ng/mL est souvent considéré comme élevé et peut nécessiter une investigation supplémentaire pour rechercher un cancer de la prostate.
*   Votre oncle a un taux de PSA total de 

In [8]:
name = list(images)[0]
print(ask("What is the PSA value on this report, with its unit?", images[name]))

(1 s, 18 tokens)
The PSA value on the report is **3.80 ng/mL**.


In [9]:
# Test A with the same comparison in text only, without the image
TEXT_QUESTIONS = [
    "Un PSA total de 3,80 ng/mL, avec une valeur de référence < 4,00 ng/mL : est-ce au-dessus ou en dessous de la limite ? Réponds en une phrase.",
    "A total PSA of 3.80 ng/mL, with a reference range < 4.00 ng/mL: is it above or below the limit? Answer in one sentence.",
    "Un PSA total de 2,50 ng/mL, avec une valeur de référence < 4,00 ng/mL : est-ce au-dessus ou en dessous de la limite ? Réponds en une phrase.",
    "A total PSA of 0.80 ng/mL, with a reference range < 4.00 ng/mL: is it above or below the limit? Answer in one sentence.",
]
for q in TEXT_QUESTIONS:
    print("Q:", q)
    print("A:", ask(q, max_new_tokens=100))
    print()

Q: Un PSA total de 3,80 ng/mL, avec une valeur de référence < 4,00 ng/mL : est-ce au-dessus ou en dessous de la limite ? Réponds en une phrase.
(2 s, 33 tokens)
A: La valeur de 3,80 ng/mL est considérée comme étant en dessous de la limite de référence de 4,00 ng/mL.

Q: A total PSA of 3.80 ng/mL, with a reference range < 4.00 ng/mL: is it above or below the limit? Answer in one sentence.
(2 s, 30 tokens)
A: The total PSA level of 3.80 ng/mL is below the reference range of < 4.00 ng/mL.

Q: Un PSA total de 2,50 ng/mL, avec une valeur de référence < 4,00 ng/mL : est-ce au-dessus ou en dessous de la limite ? Réponds en une phrase.
(2 s, 31 tokens)
A: Le PSA total de 2,50 ng/mL est en dessous de la limite de référence de 4,00 ng/mL.

Q: A total PSA of 0.80 ng/mL, with a reference range < 4.00 ng/mL: is it above or below the limit? Answer in one sentence.
(1 s, 20 tokens)
A: The total PSA level of 0.80 ng/mL is below the reference range.



In [10]:
# Test B with the image of R001 with a direct question
name = list(images)[0]
IMAGE_QUESTIONS = [
    "Le PSA total sur ce compte rendu est-il au-dessus ou en dessous de la valeur de référence ? Réponds en une phrase.",
    "Is the total PSA on this report above or below its reference range? Answer in one sentence.",
]
for q in IMAGE_QUESTIONS:
    print("Q:", q)
    print("A:", ask(q, images[name], max_new_tokens=100))
    print()

Q: Le PSA total sur ce compte rendu est-il au-dessus ou en dessous de la valeur de référence ? Réponds en une phrase.
(1 s, 17 tokens)
A: Le PSA total sur ce compte rendu est en dessous de la valeur de référence.

Q: Is the total PSA on this report above or below its reference range? Answer in one sentence.
(2 s, 31 tokens)
A: The total PSA result of 3.80 ng/mL is above the reference range of < 4.00 ng/mL.

